### ▶️ Versi Kaggle — Background Run & Simpan Weights (V3)

Notebook ini = **salinan 1:1 dari `NCCT_MultiStage_Classification_V3.ipynb`** (logika model, TTA, agregator, threshold, pipeline — TIDAK diubah),
kecuali bagian bertanda **`[KAGGLE]`** agar bisa jalan di Kaggle:

| # | Asli (Colab) | Versi Kaggle ini |
|---|---|---|
| 1 | `MODE='colab'` + `!gdown` dari Google Drive | `MODE='kaggle'` → dataset dibaca dari **`/kaggle/input/`** (upload `NCCT_fixed_split.zip` sebagai Kaggle Dataset dulu, atau attach dataset yang sudah ada). Fallback `colab`/`local` tetap tersedia |
| 2 | `OUTPUT_DIR='outputs_v3'` (ephemeral di Colab) | `OUTPUT_DIR='/kaggle/working/outputs_v3'` → otomatis muncul di tab **Output**; wajib **Save Version → Save & Run All (Commit)** agar persisten |
| 3 | Hanya simpan `*_best.pt` (tanpa resume) | **Background-safe**: tiap epoch simpan `*_last_full.pt` (model+optimizer+scheduler+scaler+epoch) secara **atomic** + auto-**resume** jika kernel restart/timeout + handler `KeyboardInterrupt` |
| 4 | Tanpa arsip weights | Cell terakhir membuat **`ncct_v3_weights.zip`** di `/kaggle/working/` → download dari tab Output / upload sebagai Dataset untuk reuse |
| 5 | `EPOCHS=1` (demo; stage 2/3 ≈ random) | Default **`EPOCHS=15`** untuk run final (ubah di cell konfigurasi) |

**Cara pakai di Kaggle (5 langkah):**
1. Upload `NCCT_fixed_split.zip` (atau folder `NCCT_fixed_split/` yang sudah di-extract) sebagai **Kaggle Dataset** → **Add Input** ke notebook ini. Atau upload zip ke dataset lalu biarkan notebook yang meng-unzip otomatis.
2. Sesuaikan `KAGGLE_DATASET_SLUG` di cell konfigurasi bila perlu (atau biarkan auto-detect — notebook mencari folder `NCCT_fixed_split` di `/kaggle/input/`).
3. **Run All**. Browser boleh ditutup — sel tetap jalan di server (limit sesi GPU ≈ 12 jam, kuota ≈ 30 jam/minggu; hemat kuota: matikan GPU saat tidak training).
4. Jika sesi timeout/interrupt → **jalankan ulang notebook yang sama**: training otomatis **resume** dari `*_last_full.pt` (tidak mengulang dari nol).
5. Setelah selesai → **Save Version → Save & Run All (Commit)** → download `ncct_v3_weights.zip` dari tab **Output**, atau **Commit to Dataset** agar bisa di-attach di sesi berikutnya (`LOAD_WEIGHTS_DIR`).

> Semua seleksi backbone/agregator/threshold tetap **HANYA dari VAL** (no leakage), sama seperti V3 asli.


# NCCT Multi-Stage Classification **V3** — Fix Laterality, Patient-Level Decision Tuning &amp; Leakage-Free Evaluation

Pipeline klasifikasi **bertingkat** pada citra CT Non-Contrast (NCCT) pasien stroke — **V3**, perbaikan menyeluruh dari V2:

```
                    ┌────────────────────────────────────────────┐
  Citra NCCT  ──►   │  STAGE 1 : Apakah pasien mengalami LVO?    │
                    └────────────────────────────────────────────┘
                              │ Ya                        │ Tidak
                              ▼                           ▼
              ┌───────────────────────────────┐      Non-LVO
              │ STAGE 2 : Laterality          │     (selesai)
              │   Left / Right                │
              ├───────────────────────────────┤
              │ STAGE 3 : Etiologi            │
              │   Cardioembolic / Non-Cardio  │
              └───────────────────────────────┘
```


## Perbaikan utama V3

| # | V2 | V3 |
|---|----------|----|
| 1 | Augmentasi seragam (termasuk **HFlip untuk laterality**) | **Augmentasi per-stage**: HFlip dimatikan utk `stage2_laterality` (bug label), aktif utk stage1/3 |
| 3 | Model terbaik dipilih berdasar **test F1** (leakage) | **Seleksi backbone/agregator/threshold HANYA dari VAL**, test dievaluasi sekali di akhir |
| 4 | Threshold 0.5 & `mean` di-hardcode | Tuning threshold + pemilihan agregator slice (`mean/median/max/vote`) di **val** |
| 5 | Tanpa TTA (Test Time Augmentation) | **Horizontal-flip TTA**; utk laterality probabilitas di-invert karena flip membalik kelas |
| 6 | Evaluasi pipeline tanpa CI (Confidence Interval) | **Bootstrap 95% CI** utk metrik level pasien (test stage 2/3 hanya ~20 pasien → varian besar) |
| 7 | Re-decode 8.6k citra tiap epoch | **LRU cache** decode+resize di Dataset |
| 8 | Output tanpa metadata konfigurasi | **Snapshot konfigurasi (JSON)** + checkpoint berisi metadata (tidak hanya state_dict) |
| 9 | Gate evaluasi stage 2/3 = prediksi stage 1 saja | Dilaporkan **prediksi-gate DAN oracle-gate** agar efek salah-klasifikasi stage 1 terlihat |

**Sumber data (FIXED SPLIT):** zip dari Google Drive (dibuat oleh `NCCT_Fixed_Dataset_Split.ipynb`):
- `metadata/{stage1_lvo,stage2_laterality,stage3_etiology}/{train,val,test}.csv`
- Folder citra `train/`, `val/`, `test/` (struktur asli dipertahankan)

**Stage:**
- **Stage 1** — LVO vs Non-LVO (semua pasien)
- **Stage 2** — Left vs Right (hanya LVO, ada label laterality)
- **Stage 3** — Cardioembolic vs Non-Cardioembolic (hanya LVO, ada label etiologi)


### 1. Install & Import Library

In [ ]:
import os, re, glob, zipfile, random, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, classification_report)

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


def set_seed(seed):
    """Set semua random seed utk reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

### 2. Konfigurasi

Ubah `MODE` menjadi `'local'` untuk uji cepat dengan sample lokal, atau `'colab'` untuk dataset penuh dari Google Drive.

> **PENTING:** dataset **tidak displit ulang**. Zip berisi *fixed split* yang sudah dibuat sekali oleh
> `NCCT_Fixed_Dataset_Split.ipynb` (seed tetap 42). Perbedaan besar dgn V2 ada di sini:
> `MONITOR`, `USE_TTA`, `STRONG_AUG`, `AGGREGATOR` (fallback) dan `NUM_WORKERS`/`CACHE_IMAGES` ditambahkan;
> default `EPOCHS` dinaikkan karena 1 epoch terbukti menghasilkan performa ≈ random utk stage 2 & 3.

In [ ]:
# ============================================================
#  KONFIGURASI V3 — [KAGGLE] MODE kaggle + background-safe
# ============================================================
IS_KAGGLE = ('/kaggle' in __import__('os').getcwd()) or __import__('os').path.exists('/kaggle')

# 'kaggle' → baca dataset dari /kaggle/input (tanpa download)
# 'colab'  → download zip FIXED SPLIT dari Google Drive (asli V3)
# 'local'  → pakai folder sample lokal NCCT_fixed_split/
MODE = 'kaggle' if IS_KAGGLE else 'colab'

GDRIVE_FILE_ID = '1PAMWLa6TAh_yFW5RY4FQxDqjIoytqgcb'   # dipakai hanya bila MODE='colab'
SPLIT_ZIP      = 'NCCT_fixed_split.zip'
SPLIT_SUBPATH  = 'NCCT_fixed_split'   # nama folder dataset di dalam /kaggle/input/<slug>/
KAGGLE_DATASET_SLUG = ''  # opsional: 'username/ncct-fixed-split'. Kosongkan = auto-detect.

# [KAGGLE] resolve SPLIT_DIR & OUTPUT_DIR sesuai environment
if IS_KAGGLE:
    import glob as _glob
    _cands = sorted(_glob.glob('/kaggle/input/*/NCCT_fixed_split'))
    if KAGGLE_DATASET_SLUG and '/' in KAGGLE_DATASET_SLUG:
        _slug_tail = KAGGLE_DATASET_SLUG.split('/')[-1]
        _cands = sorted(_glob.glob(f'/kaggle/input/{_slug_tail}/NCCT_fixed_split')) or _cands
        _cands += sorted(_glob.glob(f'/kaggle/input/{_slug_tail}/*'))
    if _cands:
        SPLIT_DIR = _cands[0]
    else:
        SPLIT_DIR = f'/kaggle/input/{SPLIT_SUBPATH}'  # akan divalidasi di cell download
    OUTPUT_DIR = '/kaggle/working/outputs_v3'
else:
    SPLIT_DIR = 'NCCT_fixed_split'
    OUTPUT_DIR = 'outputs_v3'

IMG_SIZE   = 224
BATCH_SIZE = 32

# ===== BACKBONE ABLATION =====
BACKBONES  = ['resnet18', 'resnet50', 'efficientnet_b0', 'convnext_tiny']
PRETRAINED = True

# ===== TRAINING =====
# [KAGGLE] EPOCHS=15 utk run final (V3 asli pakai 1 → stage 2/3 ≈ random).
EPOCHS        = 15
LR            = 1e-4
PATIENCE      = 6          # early stopping (epoch tanpa peningkatan metrik monitor)
WARMUP_EPOCHS = 1          # linear warmup sebelum cosine annealing
GRAD_CLIP     = 1.0        # gradient clipping (max norm)
WEIGHT_DECAY  = 1e-4
AMP           = True       # mixed precision (Kaggle P100: float16; T4: float16 optimal)

# ===== [KAGGLE] BACKGROUND-SAFE / RESUME =====
RESUME     = True   # lanjutkan dari *_last_full.pt bila ada (wajib ON utk run background)
SAVE_EVERY = 1      # simpan full-state tiap N epoch (1 = tiap epoch, aman utk limit 12 jam)

# ===== MONITOR / EARLY STOPPING =====
# 'patient_f1' → simpan model terbaik berdasar F1 level PASIEN pd val (metrik klinis)
# 'image_f1'   → simpan berdasar F1 level citra pd val (lebih stabil, perilaku V2)
MONITOR = 'patient_f1'

# ===== EVALUASI & KEPUTUSAN LEVEL PASIEN =====
PATIENT_LEVEL  = True      # selalu aktif di V3 (dipakai utk seleksi & keputusan)
AGGREGATOR     = 'mean'    # fallback; agregator terbaik dipilih otomatis pd VAL: 'mean'|'median'|'max'|'vote'
TUNE_THRESHOLD = True      # cari threshold optimal pd VALIDASI (bukan test) → no leakage
USE_TTA        = True      # test-time augmentation: horizontal-flip ensemble

# ===== DATA UTIL =====
NUM_WORKERS  = 2
CACHE_IMAGES = True        # LRU cache decode+resize citra utk epoch berulang
STRONG_AUG   = False       # True → tambah affine/translate utk augmentasi lebih kuat

# ===== REPRODUCIBILITY =====
SEEDS = [42]               # gunakan [42, 43, 44] utk mean±std antar seed

# ===== [KAGGLE] REUSE WEIGHTS DARI DATASET (opsional) =====
# Attach dataset weights lama via Add Input, mis. '/kaggle/input/ncct-v3-weights',
# lalu isi LOAD_WEIGHTS_DIR agar _best.pt dipakai sebagai warm-start / skip training.
LOAD_WEIGHTS_DIR = ''      # '' = training dari awal (atau resume dari OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# snapshot konfigurasi → disimpan ke <OUTPUT_DIR>/config_snapshot.json
CONFIG = dict(mode=MODE, gdrive_file_id=GDRIVE_FILE_ID, split_zip=SPLIT_ZIP, split_dir=SPLIT_DIR,
              kaggle_dataset_slug=KAGGLE_DATASET_SLUG, load_weights_dir=LOAD_WEIGHTS_DIR,
              img_size=IMG_SIZE, batch_size=BATCH_SIZE, backbones=BACKBONES, pretrained=PRETRAINED,
              epochs=EPOCHS, lr=LR, patience=PATIENCE, warmup_epochs=WARMUP_EPOCHS,
              grad_clip=GRAD_CLIP, weight_decay=WEIGHT_DECAY, amp=AMP, monitor=MONITOR,
              resume=RESUME, save_every=SAVE_EVERY,
              patient_level=PATIENT_LEVEL, aggregator=AGGREGATOR, tune_threshold=TUNE_THRESHOLD,
              tta=USE_TTA, num_workers=NUM_WORKERS, cache_images=CACHE_IMAGES, strong_aug=STRONG_AUG,
              seeds=SEEDS)

print('IS_KAGGLE:', IS_KAGGLE, '| Mode:', MODE, '| Device:', device)
print('SPLIT_DIR:', SPLIT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Backbones:', BACKBONES)
print('Epochs:', EPOCHS, '| Monitor:', MONITOR, '| AMP:', AMP, '| TTA:', USE_TTA, '| Resume:', RESUME)
print('Seeds:', SEEDS, '| Tuning threshold pd VAL:', TUNE_THRESHOLD)


### 3. Download Dataset Fixed Split dari Google Drive

Download zip **fixed split** (bukan dataset mentah) → hasil langsung siap dipakai, **tanpa split ulang**.

In [ ]:
# [KAGGLE] 3-mode: kaggle (/kaggle/input) | colab (gdown) | local (folder)
if MODE == 'kaggle':
    # 1) dataset sudah ter-extract di /kaggle/input/.../NCCT_fixed_split → langsung pakai
    if os.path.isdir(SPLIT_DIR) and os.path.isdir(os.path.join(SPLIT_DIR, 'metadata')):
        print('Mode kaggle → pakai dataset:', SPLIT_DIR)
        print('Isi SPLIT_DIR:', sorted(os.listdir(SPLIT_DIR)))
    else:
        # 2) fallback: cari zip di /kaggle/input atau /kaggle/working lalu extract ke /kaggle/working
        cand = ([z for z in glob.glob('/kaggle/input/**/*.zip', recursive=True) if 'fixed' in z.lower()]
                or glob.glob('/kaggle/input/**/*.zip', recursive=True)
                or ([SPLIT_ZIP] if os.path.exists(SPLIT_ZIP) else [])
                or glob.glob('*.zip'))
        print('Kandidat zip:', cand)
        if not cand:
            raise RuntimeError(
                'Dataset tidak ditemukan. Upload NCCT_fixed_split.zip sebagai Kaggle Dataset, '
                'lalu Add Input ke notebook ini (cek KAGGLE_DATASET_SLUG / SPLIT_SUBPATH).')
        zip_src = cand[0]
        SPLIT_DIR = '/kaggle/working/NCCT_fixed_split'
        import zipfile as _zf
        print(f'Extract {zip_src} → {SPLIT_DIR} ...')
        with _zf.ZipFile(zip_src) as zf:
            zf.extractall(SPLIT_DIR if zip_src.lower().endswith('fixed') else '/kaggle/working')
        # normalisasi: bila hasil extract nested (…/NCCT_fixed_split/NCCT_fixed_split), ratakan
        _nested = os.path.join(SPLIT_DIR, 'NCCT_fixed_split')
        if os.path.isdir(_nested) and not os.path.isdir(os.path.join(SPLIT_DIR, 'metadata')):
            SPLIT_DIR = _nested
        print('Isi SPLIT_DIR:', sorted(os.listdir(SPLIT_DIR)))
elif MODE == 'colab':
    get_ipython().system('pip install -q gdown')
    get_ipython().run_line_magic('gdown', GDRIVE_FILE_ID)
    zips = [z for z in glob.glob('*.zip') if 'fixed' in z.lower()] or glob.glob('*.zip')
    print('Zip hasil download:', zips)
    if not zips:
        raise RuntimeError('Tidak ada zip. Periksa GDRIVE_FILE_ID.')
    zip_src = SPLIT_ZIP if os.path.exists(SPLIT_ZIP) else zips[0]
    get_ipython().system(f'unzip -q -o {zip_src} -d NCCT_fixed_split')
    print('Isi SPLIT_DIR:', sorted(os.listdir(SPLIT_DIR)))
else:
    assert os.path.isdir(SPLIT_DIR), f'Folder {SPLIT_DIR} tidak ditemukan (mode local).'
    print('Mode local → pakai folder sample:', SPLIT_DIR)
    print('Isi:', sorted(os.listdir(SPLIT_DIR)))


### 4. Load Fixed Split

CSV metadata per stage + per set dimuat, lalu `image_path` relatif diubah menjadi absolut terhadap `SPLIT_DIR`.

- `filter_missing=True` → baris yang citranya tidak ada dibuang (agar mode `local` dengan sample tetap jalan).

In [ ]:
STAGES = ['stage1_lvo', 'stage2_laterality', 'stage3_etiology']
SPLITS = ('train', 'val', 'test')


def load_fixed_split(split_root=SPLIT_DIR, filter_missing=True):
    """Muat CSV split FIXED dari folder hasil split.

    - `image_path` relatif (mis. `train/LVO/p1_.../03.jpg`) → absolut thd split_root.
    - `filter_missing=True` → buang baris yg citranya tidak ada (utk sample lokal).
    """
    out, dropped = {}, 0
    for stage in STAGES:
        for sp in SPLITS:
            fp = os.path.join(split_root, 'metadata', stage, f'{sp}.csv')
            df = pd.read_csv(fp)
            df['image_path'] = df['image_path'].apply(
                lambda p: os.path.join(os.path.abspath(split_root), p))
            df['label'] = df['label'].astype(int)
            if filter_missing:
                n0 = len(df)
                df = df[df['image_path'].apply(os.path.exists)]
                dropped += n0 - len(df)
            out[(stage, sp)] = df.reset_index(drop=True)
    if dropped:
        print(f'[info] {dropped} baris dibuang krn citra tidak ada di folder lokal (sample).')
    return out


splits = load_fixed_split()
print('Jumlah citra & pasien per (stage, set):')
for stage in STAGES:
    sizes = [len(splits[(stage, s)]) for s in SPLITS]
    subs  = [splits[(stage, s)]['subject'].nunique() for s in SPLITS]
    print(f'  {stage:22s} citra  T:{sizes[0]:5d} V:{sizes[1]:5d} Te:{sizes[2]:5d}  '
          f'| pasien T:{subs[0]:3d} V:{subs[1]:3d} Te:{subs[2]:3d}')

# ---- Verifikasi: bebas leakage antar set & konsisten antar stage ----
for stage in STAGES:
    tr_s, va_s, te_s = [set(splits[(stage, s)]['subject']) for s in SPLITS]
    assert tr_s.isdisjoint(va_s) and tr_s.isdisjoint(te_s) and va_s.isdisjoint(te_s), \
        f'LEAKAGE di {stage}!'
    if stage != 'stage1_lvo':
        te1 = set(splits[('stage1_lvo', 'test')]['subject'])
        assert te_s.issubset(te1), f'{stage}: pasien test di luar split global!'
print('✔ Split bebas data leakage antar set & konsisten antar stage.')

### 5. Analisis Dataset

Cek distribusi kelas tiap stage & jumlah slice per pasien (dasar pentingnya evaluasi level pasien).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
stage_titles = {'stage1_lvo': 'Stage 1: LVO vs Non-LVO',
                'stage2_laterality': 'Stage 2: Laterality (hanya LVO)',
                'stage3_etiology': 'Stage 3: Etiologi (hanya LVO)'}
for ax, stage in zip(axes, STAGES):
    vc = splits[(stage, 'train')]['label'].value_counts().sort_index()
    labels = {0: '0', 1: '1'}[vc.index[0]] if len(vc) else '0'
    ax.bar(vc.index.astype(str), vc.values, color=['#3498db', '#e74c3c'])
    ax.set_title(stage_titles[stage])
    for i, v in enumerate(vc.values):
        ax.annotate(str(int(v)), (i, v), ha='center', va='bottom')
    ax.set_ylabel('Jumlah citra (train)')
plt.tight_layout(); plt.show()

# Slice per pasien → evaluasi level pasien lebih bermakna
tr1 = splits[('stage1_lvo', 'train')]
spp = tr1.groupby('subject').size()
print(f'Slice/pasien (train stage1): mean={spp.mean():.1f}  median={spp.median():.0f}  max={spp.max()}')
plt.figure(figsize=(8, 3))
sns.histplot(spp, bins=30)
plt.xlabel('Jumlah slice per pasien')
plt.title('Distribusi slice/pasien (train stage 1)')
plt.show()

# Contoh citra tiap kelas
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, (title, lab) in enumerate([('LVO (selected_ncct)', 1), ('Non-LVO (lacunar)', 0)]):
    sub = tr1[tr1['label'] == lab]
    p = sub.sample(1, random_state=42 + i)['image_path'].iloc[0]
    img = Image.open(p).convert('RGB')
    axes[i].imshow(img)
    axes[i].set_title(f'{title}\n{os.path.basename(p)}', fontsize=9)
    axes[i].axis('off')
plt.tight_layout(); plt.show()

### 6. Dataset & Dataloader — Augmentasi PER-STAGE + LRU Cache

**Perbaikan #1 (kritis):** augmentasi sekarang dibangun per stage.

- **`stage2_laterality` TIDAK memakai `RandomHorizontalFlip`** — mem-flip citra akan membalik
  makna label (kiri ↔ kanan). V2 memakai transform yang sama utk semua stage sehingga label
  stage 2 menjadi ~50% noise → model tidak bisa belajar (hasil ≈ chance).
- Stage 1 & 3 aman terhadap flip (kelas tidak bergantung sisi).
- Ditambahkan **LRU cache** (`CACHE_IMAGES`) agar citra yang sudah di-decode+resize tidak dibuka
  ulang dari disk setiap epoch (8.6k citra train stage 1 × N epoch).

Setiap sample mengembalikan `(image, label, subject, image_path)` — `subject` diperlukan untuk evaluasi level pasien.

In [ ]:
class StrokeImageDataset(Dataset):
    """Dataset citra stroke dari CSV fixed split: 1 sample = 1 citra + label + subject.

    cache=True → LRU cache hasil decode+resize agar epoch berikutnya lebih cepat
    (citra dibuka sekali, transform acak tetap diterapkan tiap akses).
    """
    def __init__(self, df, transform=None, cache=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self._cache = OrderedDict() if cache else None
        self._CACHE_MAX = 2000   # ~2000 x 224x224 RGB ≈ ratusan MB — batas aman utk Colab

    def __len__(self):
        return len(self.df)

    def _load(self, path):
        if self._cache is None:
            return Image.open(path).convert('RGB')
        if path in self._cache:
            self._cache.move_to_end(path)
            return self._cache[path]
        img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        if len(self._cache) >= self._CACHE_MAX:
            self._cache.popitem(last=False)
        self._cache[path] = img
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load(row['image_path'])
        if self.transform is not None:
            img = self.transform(img)
        return img, int(row['label']), row['subject'], row['image_path']


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def _eval_tf():
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def build_train_tf(stage, strong=False):
    """Augmentasi PER-STAGE.

    Kunci (fix bug V2): RandomHorizontalFlip TIDAK boleh dipakai utk stage2_laterality —
    mem-flip citra membalik label Left<->Right sehingga 50% label menjadi noise.
    Stage 1 (LVO) & 3 (etiologi) aman terhadap flip.
    """
    hflip = (stage != 'stage2_laterality')
    ops = [transforms.Resize((IMG_SIZE, IMG_SIZE)),
           transforms.RandomRotation(degrees=5)]
    if strong:
        ops.append(transforms.RandomAffine(degrees=0, translate=(0.06, 0.06), scale=(0.94, 1.06)))
    ops += [transforms.RandomHorizontalFlip(p=0.5 if hflip else 0.0),
            # brightness/contrast jitter aman utk citra grayscale medis
            transforms.ColorJitter(brightness=0.1, contrast=0.15),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]
    return transforms.Compose(ops)


def make_loaders(stage, batch_size=BATCH_SIZE, cache=CACHE_IMAGES):
    """Loader utk satu stage dari FIXED SPLIT (tidak ada split ulang)."""
    tf_tr = build_train_tf(stage, strong=STRONG_AUG)
    tf_ev = _eval_tf()
    tr_ds = StrokeImageDataset(splits[(stage, 'train')], tf_tr, cache=cache)
    va_ds = StrokeImageDataset(splits[(stage, 'val')],   tf_ev, cache=cache)
    te_ds = StrokeImageDataset(splits[(stage, 'test')],  tf_ev, cache=cache)
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    va_ld = DataLoader(va_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    te_ld = DataLoader(te_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    return tr_ld, va_ld, te_ld

### 7. Fungsi Training, Evaluasi (dgn TTA) & Utilitas Keputusan Level Pasien

Peningkatan fungsi dibanding V2:

- **`evaluate_model(..., tta, invert_flip)`** — dukungan **test-time augmentation** horizontal-flip.
  - `tta=True` → probabilitas = rata-rata `p` dan `p_flipped`.
  - `invert_flip=True` → khusus **laterality**: karena citra ter-flip membalik kelas, probabilitas
    di-invert (`p_f[:, [1, 0]]`) sebelum dirata-rata. Dilarang utk stage lain.
- **Utilitas level pasien** — `scores_df`, `patient_df` (agregator `mean/median/max/vote`),
  `patient_metrics_from_sdf`, dan `tune_threshold` (scanning threshold optimal).
  Threshold & agregator HARUS ditentukan dari **val**, bukan test.
- **`train_model(..., monitor)`** — `monitor='patient_f1'` memilih epoch terbaik berdasar **F1
  level pasien pd val** (metrik relevan klinis); `monitor='image_f1'` mempertahankan perilaku V2.
  Checkpoint kini menyimpan `{'model': state_dict, 'meta': {...}}`.

In [ ]:
def build_model(num_classes=2, backbone='resnet18', pretrained=PRETRAINED):
    """Model CNN pretrained dari timm (utk ablation backbone)."""
    try:
        model = timm.create_model(backbone, pretrained=pretrained, num_classes=num_classes)
    except TypeError:
        weights = 'IMAGENET1K_V1' if pretrained else None
        model = timm.create_model(backbone, weights=weights, num_classes=num_classes)
    return model


def compute_metrics(y_true, y_pred, y_prob):
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'auc':       roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan'),
    }


@torch.no_grad()
def evaluate_model(model, loader, criterion=None, tta=False, invert_flip=False):
    """Evaluasi pada loader. Return (metrics, records) — records per-slice:
    [(subject, y_true, y_pred, y_prob), ...] → dipakai utk level pasien.

    tta=True         → horizontal-flip ensemble: probs = mean(p, p_flip).
    invert_flip=True → HANYA utk laterality (stage2): citra ter-flip membalik kelas,
                       jadi prob kelas di-invert sebelum dirata-rata. JANGAN utk stage lain.
    """
    model.eval()
    records, loss_sum, n = [], 0.0, 0
    for xb, yb, subj, _ in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)
        if tta:
            xf = torch.flip(xb, dims=[3])                       # horizontal flip (N,C,H,W)
            p_f = torch.softmax(model(xf), dim=1)
            if invert_flip:
                p_f = p_f[:, [1, 0]]                            # balik makna kelas krn citra ter-flip
            probs = (probs + p_f) / 2.0
        records += [(s, int(yt), int(yp), float(pr)) for s, yt, yp, pr in
                    zip(subj, yb.tolist(), probs.argmax(1).tolist(), probs[:, 1].tolist())]
        if criterion is not None:
            loss_sum += criterion(logits, yb).item() * len(yb)
            n += len(yb)
    y_true = [r[1] for r in records]
    y_pred = [r[2] for r in records]
    y_prob = [r[3] for r in records]
    metrics = compute_metrics(y_true, y_pred, y_prob)
    if criterion is not None:
        metrics['loss'] = loss_sum / max(n, 1)
    return metrics, records


# ============ UTILITAS LEVEL PASIEN & KEPUTUSAN ============

def scores_df(records):
    """DataFrame per-slice: subject, y_true, y_pred, y_prob."""
    return pd.DataFrame(records, columns=['subject', 'y_true', 'y_pred', 'y_prob'])


def patient_df(sdf, aggregator='mean'):
    """Agregasi slice → 1 baris/pasien (prob kelas-1).

    aggregator:
      'mean'   → rata-rata probabilitas seluruh slice (default V2)
      'median' → median prob (tahan outlier slice)
      'max'    → prob slice tertinggi (sesuai lesi yang tampil hanya di bbrp slice)
      'vote'   → fraksi slice 'positif' (p>=0.5) → keputusan mayoritas slice
    """
    grp = sdf.groupby('subject')['y_prob']
    if aggregator == 'median':
        p = grp.median()
    elif aggregator == 'max':
        p = grp.max()
    elif aggregator == 'vote':
        p = grp.apply(lambda x: float((x >= 0.5).mean()))
    else:
        p = grp.mean()
    pdf = p.reset_index().rename(columns={'y_prob': 'p'})
    yt = sdf.drop_duplicates('subject').set_index('subject')['y_true']
    pdf['y_true'] = pdf['subject'].map(yt)
    return pdf


def patient_metrics_from_sdf(sdf, aggregator='mean', threshold=0.5):
    """Metrik level pasien dari records per-slice."""
    pdf = patient_df(sdf, aggregator)
    yp = (pdf['p'] >= threshold).astype(int)
    m = compute_metrics(pdf['y_true'].tolist(), yp.tolist(), pdf['p'].tolist())
    m['n'] = int(len(pdf))
    return m, pdf


def tune_threshold(sdf, aggregator='mean', metric='f1', step=0.02):
    """Scan threshold optimal pd data VALIDASI (JANGAN dipakai pd test — leakage!).

    Return (best_threshold, best_metric_value, curve). Untuk LVO-triage yg mengejar recall,
    ganti metric='recall' atau baca kurva di bagian 11.
    """
    pdf = patient_df(sdf, aggregator)
    best_t, best_m, curve = 0.5, -1.0, []
    for t in np.arange(0.0, 1.0 + 1e-9, step):
        yp = (pdf['p'] >= t).astype(int)
        m = compute_metrics(pdf['y_true'].tolist(), yp.tolist(), pdf['p'].tolist())
        curve.append((float(t), float(m[metric])))
        if m[metric] > best_m:
            best_t, best_m = float(t), float(m[metric])
    return best_t, best_m, curve


def save_atomic(payload, path, save_fn=None):
    """[KAGGLE] Simpan atomic: tulis ke .tmp lalu os.replace (anti-korup saat kernel di-kill).

    Pola produksi timm.CheckpointSaver._replace — wajib untuk background run Kaggle
    (time limit bisa mematikan proses di tengah torch.save).
    """
    import os as _os, tempfile as _tf
    _d = _os.path.dirname(path) or '.'
    _os.makedirs(_d, exist_ok=True)
    fd, tmp = _tf.mkstemp(dir=_d, prefix='tmp_ckpt_')
    _os.close(fd)
    try:
        (save_fn or __import__('torch').save)(payload, tmp)
        _os.replace(tmp, path)   # atomic di POSIX
    finally:
        if _os.path.exists(tmp):
            try:
                _os.remove(tmp)
            except OSError:
                pass


def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, patience=PATIENCE,
                label='', class_weights=None, seed=42, amp=AMP, monitor=MONITOR, tta=USE_TTA,
                resume=RESUME):
    """Training: warmup+cosine, AMP, grad clip, early stopping + [KAGGLE] background-safe.

    Tambahan [KAGGLE] dibanding V3 asli:
    - Tiap epoch simpan FULL-STATE `{label}_last_full.pt` (model+optimizer+scheduler+scaler+
      epoch+best_score+history) secara atomic → tahan terhadap timeout/interrupt Kaggle.
    - `resume=True` → lanjut otomatis dari `_last_full.pt` (tidak mengulang dari epoch 0).
    - `try/except KeyboardInterrupt` → simpan `_interrupted_full.pt` sebelum keluar.
    - Checkpoint BEST `{label}_best.pt` tetap format V3 asli ({'model','meta'}) agar kompatibel
      dengan `load_best_model` di bagian pipeline.
    """
    set_seed(seed)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    if WARMUP_EPOCHS > 0 and WARMUP_EPOCHS < epochs:
        warmup = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
        cosine = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - WARMUP_EPOCHS)
        scheduler = optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine],
                                                    milestones=[WARMUP_EPOCHS])
    else:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    invert = ('stage2_laterality' in label)          # flip TTA membalik kelas utk laterality
    use_amp = amp and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    history = {'train_loss': [], 'val_loss': [], 'val_img_f1': [], 'val_pat_f1': []}
    best_score, best_epoch, bad_epochs = -1.0, 0, 0
    save_path = os.path.join(OUTPUT_DIR, f'{label}_best.pt')
    last_path = os.path.join(OUTPUT_DIR, f'{label}_last_full.pt')
    start_epoch = 1

    # ---- [KAGGLE] resume dari full-state bila ada ----
    if resume and os.path.exists(last_path):
        try:
            ck = torch.load(last_path, map_location=device, weights_only=False)
            model.load_state_dict(ck['model'])
            optimizer.load_state_dict(ck['optimizer'])
            if 'scheduler' in ck and ck['scheduler'] is not None:
                try:
                    scheduler.load_state_dict(ck['scheduler'])
                except Exception as e:
                    print(f'[{label}] peringatan: scheduler state tidak cocok, lanjut tanpa itu ({e})')
            if 'scaler' in ck and ck['scaler'] is not None and use_amp:
                try:
                    scaler.load_state_dict(ck['scaler'])
                except Exception as e:
                    print(f'[{label}] peringatan: scaler state tidak cocok ({e})')
            best_score = ck.get('best_score', best_score)
            best_epoch = ck.get('best_epoch', 0)
            bad_epochs = ck.get('bad_epochs', 0)
            history = ck.get('history', history)
            start_epoch = int(ck.get('epoch', 0)) + 1
            print(f'[{label}] ▶ resume dari epoch {start_epoch} (best {monitor}={best_score:.4f} @epoch {best_epoch})')
        except Exception as e:
            print(f'[{label}] peringatan: gagal resume dari {last_path} ({e}) → mulai dari awal.')

    # ---- [KAGGLE] warm-start dari weights dataset lama (opsional, hanya bila belum ada resume) ----
    if start_epoch == 1 and LOAD_WEIGHTS_DIR:
        _w = os.path.join(LOAD_WEIGHTS_DIR, f'{label}_best.pt')
        if os.path.exists(_w):
            try:
                _ck = torch.load(_w, map_location=device, weights_only=False)
                _sd = _ck['model'] if isinstance(_ck, dict) and 'model' in _ck else _ck
                model.load_state_dict(_sd)
                print(f'[{label}] warm-start dari {_w}')
            except Exception as e:
                print(f'[{label}] peringatan: gagal warm-start ({e})')

    try:
        for epoch in range(start_epoch, epochs + 1):
            model.train()
            tot_loss, tot = 0.0, 0
            for xb, yb, _, _ in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda', enabled=use_amp):
                    logits = model(xb)
                    loss = criterion(logits, yb)
                scaler.scale(loss).backward()
                if GRAD_CLIP:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                tot_loss += loss.item() * len(yb)
                tot += len(yb)
            scheduler.step()

            val_metrics, val_recs = evaluate_model(model, val_loader, criterion=criterion,
                                                   tta=tta, invert_flip=invert)
            pa_val, _ = patient_metrics_from_sdf(scores_df(val_recs), 'mean', 0.5)
            score = pa_val['f1'] if monitor == 'patient_f1' else val_metrics['f1']
            tl = tot_loss / max(tot, 1)
            history['train_loss'].append(tl)
            history['val_loss'].append(val_metrics.get('loss', float('nan')))
            history['val_img_f1'].append(val_metrics['f1'])
            history['val_pat_f1'].append(pa_val['f1'])

            print(f'[{label}] epoch {epoch:02d}/{epochs} | train_loss={tl:.4f} '
                  f'| val_img_f1={val_metrics["f1"]:.4f} | val_pat_f1={pa_val["f1"]:.4f}', flush=True)

            if score > best_score:
                best_score, best_epoch, bad_epochs = score, epoch, 0
                save_atomic({'model': model.state_dict(),
                             'meta': dict(label=label, seed=seed, monitor=monitor,
                                          epoch=epoch, val_pat_f1=pa_val['f1'],
                                          val_img_f1=val_metrics['f1'], config=CONFIG)},
                            save_path)
            else:
                bad_epochs += 1
                # [KAGGLE] simpan full-state tiap SAVE_EVERY epoch (atomic) — walau bukan best
                if (epoch % SAVE_EVERY == 0) or (bad_epochs >= patience):
                    save_atomic({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                                 'scheduler': scheduler.state_dict(),
                                 'scaler': scaler.state_dict() if use_amp else None,
                                 'epoch': epoch, 'best_score': best_score, 'best_epoch': best_epoch,
                                 'bad_epochs': bad_epochs, 'history': history,
                                 'meta': dict(label=label, seed=seed, monitor=monitor, config=CONFIG)},
                                last_path)
                if bad_epochs >= patience:
                    print(f'  >> Early stop di epoch {epoch} (best {monitor}={best_score:.4f} di epoch {best_epoch})')
                    break
            # [KAGGLE] pastikan last_full selalu ada tiap epoch (untuk resume kapan pun)
            if epoch % SAVE_EVERY == 0:
                save_atomic({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                             'scheduler': scheduler.state_dict(),
                             'scaler': scaler.state_dict() if use_amp else None,
                             'epoch': epoch, 'best_score': best_score, 'best_epoch': best_epoch,
                             'bad_epochs': bad_epochs, 'history': history,
                             'meta': dict(label=label, seed=seed, monitor=monitor, config=CONFIG)},
                            last_path)
    except KeyboardInterrupt:
        # [KAGGLE] interrupt (stop manual / pre-timeout) → selamatkan state sebelum keluar
        try:
            save_atomic({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                         'scheduler': scheduler.state_dict(),
                         'scaler': scaler.state_dict() if use_amp else None,
                         'epoch': epoch, 'best_score': best_score, 'best_epoch': best_epoch,
                         'bad_epochs': bad_epochs, 'history': history,
                         'meta': dict(label=label, seed=seed, monitor=monitor, config=CONFIG)},
                        os.path.join(OUTPUT_DIR, f'{label}_interrupted_full.pt'))
            print(f'[{label}] ⚠ interrupted di epoch {epoch} — state tersimpan, jalankan ulang utk resume.')
        except Exception as e:
            print(f'[{label}] gagal menyimpan interrupted state: {e}')
        raise

    if os.path.exists(save_path):
        _best = torch.load(save_path, map_location=device, weights_only=False)
        model.load_state_dict(_best['model'] if isinstance(_best, dict) and 'model' in _best else _best)
    print(f'[{label}] Selesai. Best {monitor} = {best_score:.4f} (epoch {best_epoch})')
    return model, history


### 8. Ablation Runner & Penyimpanan Hasil

Framework ablation (stage × backbone × seed) pada fixed split yang sama:

- Setiap run menyimpan: metrik **val** (`val_img`, `val_patient`) + **`val_scores`** dan **`test_scores`**
  (DataFrame per-slice) — dipakai utk seleksi & tuning keputusan **di val** (tanpa leakage).
- Metrik default level pasien (mean, thr 0.5) tetap dilaporkan agar bisa dibandingkan dgn V2.
- Otomatis menyimpan JSON/CSV/Markdown + **snapshot konfigurasi**.

In [ ]:
def class_weights_for(stage):
    tr = splits[(stage, 'train')]
    counts = np.bincount(tr['label'].values)
    cw = torch.tensor(len(tr) / (2 * counts), dtype=torch.float32).to(device)
    return cw, counts


def run_ablation(backbones=None, seeds=None, stages=None, epochs=None, tta=USE_TTA):
    """Ablation backbone × stage × seed pada fixed split.

    Per run menyimpan:
      - metrik VAL & TEST level citra ('val_img', 'test')
      - metrik level pasien default ('val_patient', 'patient')  → mean, thr 0.5
      - 'val_scores' & 'test_scores': DataFrame per-slice utk tuning keputusan di VAL
    Seleksi akhir model/threshold TIDAK boleh memakai kolom test (lihat bagian 11).
    """
    backbones = backbones or BACKBONES
    seeds     = seeds or SEEDS
    stages    = stages or STAGES
    epochs    = epochs or EPOCHS
    results, t0 = {}, time.time()

    for stage in stages:
        cw, counts = class_weights_for(stage)
        print(f'\n{"="*72}\nSTAGE {stage} | distribusi train: {dict(zip([0, 1], counts))}\n{"="*72}')
        tr_ld, va_ld, te_ld = make_loaders(stage)
        print(f'Citra train: {len(tr_ld.dataset)} | val: {len(va_ld.dataset)} | test: {len(te_ld.dataset)}')
        invert = (stage == 'stage2_laterality')
        results[stage] = {}
        for bb in backbones:
            results[stage][bb] = {}
            for seed in seeds:
                set_seed(seed)
                model = build_model(2, bb).to(device)
                model, hist = train_model(model, tr_ld, va_ld, epochs=epochs,
                                          label=f'{stage}_{bb}', class_weights=cw, seed=seed)
                # ---- VAL: utk seleksi & tuning keputusan (no leakage) ----
                va_metrics, va_recs = evaluate_model(model, va_ld, tta=tta, invert_flip=invert)
                pa_val, _ = patient_metrics_from_sdf(scores_df(va_recs), 'mean', 0.5)
                # ---- TEST: hanya utk laporan akhir (dipakai SEKALI) ----
                te_metrics, te_recs = evaluate_model(model, te_ld, tta=tta, invert_flip=invert)
                pa_te, _ = patient_metrics_from_sdf(scores_df(te_recs), 'mean', 0.5)
                results[stage][bb][seed] = {
                    'val_img': va_metrics, 'val_patient': pa_val, 'val_scores': scores_df(va_recs),
                    'test': te_metrics, 'patient': pa_te, 'test_scores': scores_df(te_recs),
                    'history': hist,
                }
                print(f'  ✔ {bb} (seed={seed}) | VAL img_f1={va_metrics["f1"]:.4f} pat_f1={pa_val["f1"]:.4f} '
                      f'| TEST img_f1={te_metrics["f1"]:.4f} pat_f1={pa_te["f1"]:.4f}')

    print(f'\n⏱ Total waktu: {(time.time() - t0) / 60:.1f} menit')
    return results


# ---------- Tabel & penyimpanan hasil ----------

def fmt_mean_std(values):
    vals = [v for v in values if v is not None and not (isinstance(v, float) and np.isnan(v))]
    if not vals:
        return 'N/A'
    return f'{np.mean(vals):.4f}±{np.std(vals):.4f}'


def ablation_table(results, level='test'):
    """Tabel markdown per stage: backbone × metrik (mean±std antar seed).
    level ∈ {'test', 'patient', 'val_patient', 'val_img'}."""
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    lines = []
    for stage in results:
        lines.append(f'\n### {stage} — level **{level}**')
        lines.append('| Backbone | ' + ' | '.join(m.capitalize() for m in metrics) + ' |')
        lines.append('|---|' + '---|' * len(metrics))
        for bb in results[stage]:
            cells = []
            for m in metrics:
                vals = [r[level][m] for r in results[stage][bb].values()
                        if r[level] is not None and m in r[level]]
                cells.append(fmt_mean_std(vals))
            lines.append(f'| {bb} | ' + ' | '.join(cells) + ' |')
    return '\n'.join(lines)


def save_ablation_results(results, tag=''):
    """Simpan hasil ablation ke JSON + CSV + Markdown + config snapshot di OUTPUT_DIR."""
    ts = time.strftime('%Y%m%d_%H%M%S')
    STRIP = {'history', 'val_scores', 'test_scores'}      # jangan simpan objek besar
    slim = {stage: {bb: {str(sd): {k: v for k, v in r.items() if k not in STRIP}
                         for sd, r in sdct.items()}
                    for bb, sdct in stct.items()}
            for stage, stct in results.items()}
    json_path = os.path.join(OUTPUT_DIR, f'ablation_results{tag}_{ts}.json')
    with open(json_path, 'w') as f:
        json.dump(slim, f, indent=2, default=str)

    rows = []
    for stage in results:
        for bb in results[stage]:
            for level in ['test', 'patient', 'val_patient', 'val_img']:
                first = next(iter(results[stage][bb].values()))
                if first[level] is None:
                    continue
                row = {'stage': stage, 'backbone': bb, 'level': level}
                for m in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
                    vals = [r[level][m] for r in results[stage][bb].values()
                            if r[level] is not None and m in r[level]]
                    row[m] = fmt_mean_std(vals)
                rows.append(row)
    csv_path = os.path.join(OUTPUT_DIR, f'ablation_summary{tag}_{ts}.csv')
    pd.DataFrame(rows).to_csv(csv_path, index=False)

    md_path = os.path.join(OUTPUT_DIR, f'ablation_report{tag}_{ts}.md')
    with open(md_path, 'w') as f:
        f.write('# NCCT Multi-Stage V3 — Ablation Backbone\n\n')
        f.write('Metrik default = agregasi mean + threshold 0.5 (baseline, bukan keputusan akhir).\n')
        f.write(ablation_table(results, 'test'))
        f.write('\n\n' + ablation_table(results, 'patient'))
        f.write('\n\n' + ablation_table(results, 'val_patient'))
    cfg_path = os.path.join(OUTPUT_DIR, 'config_snapshot.json')
    with open(cfg_path, 'w') as f:
        json.dump(CONFIG, f, indent=2, default=str)
    print('Tersimpan:')
    for p in (json_path, csv_path, md_path, cfg_path):
        print('  ', p)


def plot_ablation_bars(results, level='test', metric='f1'):
    """Bar chart per stage utk satu metrik (mean ± std antar seed)."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    for ax, stage in zip(axes, STAGES):
        if stage not in results:
            continue
        names, means, stds = [], [], []
        for bb in results[stage]:
            vals = [r[level][metric] for r in results[stage][bb].values()
                    if r[level] is not None and metric in r[level]]
            names.append(bb)
            means.append(np.mean(vals))
            stds.append(np.std(vals))
        bars = ax.bar(names, means, yerr=stds, capsize=4, color='#3498db')
        for b, v in zip(bars, means):
            ax.annotate(f'{v:.3f}', (b.get_x() + b.get_width() / 2, v),
                        ha='center', va='bottom', fontsize=8)
        ax.set_title(f'{stage}\n{metric.upper()} ({level})')
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()

In [ ]:
# [KAGGLE] Helper background-run: arsip weights + status sesi
import time as _time
_SESS_T0 = _time.time()

def kaggle_session_status():
    el = (_time.time() - _SESS_T0) / 3600
    print(f'Sesi berjalan {el:.2f} jam (limit GPU Kaggle ≈ 12 jam/sesi, kuota ≈ 30 jam/minggu).')
    print(f'OUTPUT_DIR: {OUTPUT_DIR}')
    pts = sorted(__import__('glob').glob(os.path.join(OUTPUT_DIR, '*.pt')))
    print(f'Checkpoint (*.pt): {len(pts)} file')
    for p in pts:
        print(f'  - {os.path.basename(p)} ({os.path.getsize(p)/1e6:.1f} MB)')

def zip_kaggle_outputs(zip_name='/kaggle/working/ncct_v3_weights.zip'):
    """[KAGGLE] Arsipkan SEMUA weights + keputusan + laporan ke 1 zip di /kaggle/working.

    Jalankan di cell terakhir (atau kapan pun). File muncul di tab Output Kaggle →
    download langsung, atau Save Version → Commit to Dataset untuk reuse.
    """
    import zipfile, glob
    pats = ['*.pt', '*.json', '*.csv', '*.md']
    files = []
    for p in pats:
        files += glob.glob(os.path.join(OUTPUT_DIR, p))
    if not files:
        print(f'Tidak ada file di {OUTPUT_DIR} — jalankan training dulu.')
        return None
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fp in sorted(files):
            zf.write(fp, arcname=f'outputs_v3/{os.path.basename(fp)}')
    print(f'✔ {zip_name} ({os.path.getsize(zip_name)/1e6:.1f} MB, {len(files)} file)')
    print('Download dari tab Output (kanan) → atau Save Version → Commit to Dataset.')
    return zip_name

kaggle_session_status()


### 9. Menjalankan Ablation

> **Estimasi waktu (Kaggle GPU P100/T4):** 1 epoch stage1 ≈ 1.5–4 menit tergantung backbone (GPU Kaggle).\n> **[KAGGLE]** Browser boleh ditutup — training lanjut di server. Batas sesi ≈ 12 jam; bila terputus, jalankan ulang notebook (auto-resume dari `*_last_full.pt`). Setelah selesai jalankan cell 17 + **Save Version** agar weights tersimpan.
> Total ≈ jumlah stage × backbone × seed × epoch efektif (early stopping).
> Untuk uji cepat, batasi variabel `RUN_*` di bawah (`RUN_EPOCHS=2` misalnya), lalu jalankan
> ulang dengan epoch penuh utk hasil final.

In [ ]:
# ============================================================
#  KONFIGURASI RUN 
# ============================================================
RUN_STAGES    = STAGES            # mis. ['stage1_lvo'] utk coba 1 stage dulu
RUN_BACKBONES = BACKBONES         # mis. ['resnet18', 'convnext_tiny']
RUN_SEEDS     = SEEDS             # mis. [42]
RUN_EPOCHS    = EPOCHS            

results = run_ablation(backbones=RUN_BACKBONES, seeds=RUN_SEEDS,
                       stages=RUN_STAGES, epochs=RUN_EPOCHS)

### 10. Hasil Ablation — Tabel Default (mean prob, threshold 0.5)

Tabel per-backbone ini **belum** memakai hasil tuning — dipakai utk melihat perbandingan
backbone apa adanya. Keputusan akhir (backbone + agregator + threshold) dipilih di bagian 11
**hanya dari VAL**, lalu test dievaluasi sekali di bagian 11/13.

In [ ]:
print(ablation_table(results, 'test'))
print(ablation_table(results, 'patient'))
print(ablation_table(results, 'val_patient'))

# Simpan baseline + snapshot konfigurasi (keputusan akhir menyusul di bagian 11)
save_ablation_results(results)

plot_ablation_bars(results, 'test', 'f1')
plot_ablation_bars(results, 'patient', 'f1')

### 11. Seleksi Model, Agregator & Threshold — HANYA dari VALIDASI (No Leakage)

Ini koreksi metodologis utama V3. Di V2, "backbone terbaik" dipilih berdasar **test F1**
kemudian dilaporkan ulang pada test yang sama → angka terlalu optimis. Di V3 urutannya:

1. **Backbone** → dipilih per stage dari F1 level-pasien pada **val** (rata-rata antar seed).
2. **Agregator slice** (`mean`/`median`/`max`/`vote`) → dipilih pada **val** utk backbone tsb.
3. **Threshold** → di-scan pada **val** (memaksimalkan F1 level pasien).

> Catatan klinis: untuk LVO sebagai alat triase, `recall` lebih penting daripada `accuracy`
> (membiarkan LVO lolos = berbahaya). Threshold maksimal-F1 adalah titik netral;
> lihat kurva threshold utk memilih operating point lain bila perlu.

In [ ]:
# ============================================================
#  SELEKSI & KEPUTUSAN — HANYA VALIDASI (no data leakage)
# ============================================================
# 1) Backbone terbaik per stage = mean F1 level-pasien pd VAL antar seed.
# 2) Agregator terbaik = max F1 level-pasien (thr 0.5) pd VAL utk backbone tsb.
# 3) Threshold optimal = scan pd VAL (maks. F1).
# TEST SET DIPAKAI SEKALI DI BAGIAN 13 — TIDAK untuk seleksi apa pun di sini.
# ------------------------------------------------------------
RUN_SEED = RUN_SEEDS[0]

def mean_over_seeds(results, stage, bb, key, metric):
    vals = [r[key][metric] for r in results[stage][bb].values()
            if r[key] is not None and metric in r[key]]
    return float(np.mean(vals)) if vals else -1.0

AGG_CHOICES = ['mean', 'median', 'max', 'vote']

# ---- 1) backbone ----
best_bb = {}
for stage in results:
    best_bb[stage] = max(results[stage],
                         key=lambda bb: mean_over_seeds(results, stage, bb, 'val_patient', 'f1'))
print('1) Backbone terbaik per stage (F1 level PASIEN, VAL):')
for s, b in best_bb.items():
    print(f'     {s:22s} → {b}')

# ---- 2) agregator ----
agg_best, agg_scores = {}, {}
for stage in results:
    sdf = results[stage][best_bb[stage]][RUN_SEED]['val_scores']
    scores = {a: patient_metrics_from_sdf(sdf, a, 0.5)[0]['f1'] for a in AGG_CHOICES}
    agg_best[stage] = max(scores, key=scores.get)
    agg_scores[stage] = scores
print('\n2) Agregator slice terbaik (thr 0.5, VAL):')
for s, a in agg_best.items():
    print(f'     {s:22s} → {a}   { {k: round(v, 3) for k, v in agg_scores[s].items()} }')

# ---- 3) threshold ----
thresholds, curves = {}, {}
if TUNE_THRESHOLD:
    for stage in results:
        sdf = results[stage][best_bb[stage]][RUN_SEED]['val_scores']
        thr, best_f1, curve = tune_threshold(sdf, agg_best[stage], 'f1')
        thresholds[stage] = float(thr)
        curves[stage] = curve
else:
    for stage in results:
        thresholds[stage], curves[stage] = 0.5, None
print('\n3) Threshold optimal (VAL, maks F1 level pasien):')
for s, t in thresholds.items():
    print(f'     {s:22s} → {t:.2f}')

DECISIONS = {s: {'backbone': best_bb[s], 'aggregator': agg_best[s], 'threshold': thresholds[s]}
             for s in results}

# ---- Kurva threshold (F1 vs threshold) pd VAL ----
fig, axes = plt.subplots(1, len(results), figsize=(5.5 * len(results), 4))
if len(results) == 1:
    axes = [axes]
for ax, stage in zip(axes, results):
    c = curves[stage]
    ax.plot([t for t, _ in c], [v for _, v in c], '-o', ms=3)
    ax.axvline(thresholds[stage], ls='--', color='red', alpha=0.7)
    ax.set_title(f'{stage} — F1 level pasien (VAL)\nthr terbaik = {thresholds[stage]:.2f}')
    ax.set_xlabel('threshold'); ax.set_ylabel('F1')
plt.tight_layout(); plt.show()

# simpan keputusan utk dipakai pipeline (bagian 12-13) & arsip
dec_path = os.path.join(OUTPUT_DIR, f'decisions_{time.strftime("%Y%m%d_%H%M%S")}.json')
with open(dec_path, 'w') as f:
    json.dump(DECISIONS, f, indent=2)
pd.DataFrame([{'stage': s, **d} for s, d in DECISIONS.items()]).to_csv(
    os.path.join(OUTPUT_DIR, 'decisions_summary.csv'), index=False)
print('\nKeputusan tersimpan:', dec_path)
print('\n⚠ Metrik TEST final dilaporkan SEKALI di bagian 13 — set ini tidak dipakai utk seleksi apa pun.')

### 12. Pipeline Multi-Stage End-to-End (Level Pasien)

Pipeline klinis: 1 pasien (banyak slice) → **Stage 1** (LVO/Non-LVO) → jika LVO lanjut
**Stage 2** (laterality) & **Stage 3** (etiologi).

Keputusan per pasien = agregasi probabilitas seluruh slice-nya (agregator hasil tuning) yang
dibandingkan terhadap **threshold hasil tuning** — bukan asumsi 0.5.

In [ ]:
assert set(STAGES).issubset(DECISIONS), (
    'Pipeline end-to-end butuh SEMUA stage. Jalankan bagian 9 dengan RUN_STAGES = STAGES.')


def load_best_model(stage, backbone):
    """Muat checkpoint V3: {'model': state_dict, 'meta': {...}}."""
    m = build_model(2, backbone).to(device)
    fp = os.path.join(OUTPUT_DIR, f'{stage}_{backbone}_best.pt')
    ckpt = torch.load(fp, map_location=device)
    sd = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
    m.load_state_dict(sd)
    m.eval()
    return m


models = {stage: load_best_model(stage, DECISIONS[stage]['backbone']) for stage in results}
print('Model terbaik dimuat (seleksi berdasar VAL):')
for stage, d in DECISIONS.items():
    print(f'  {stage:22s} → {d["backbone"]} (agg={d["aggregator"]}, thr={d["threshold"]:.2f})')


@torch.no_grad()
def slice_pos_prob(df, model, invert_flip=False, tta=USE_TTA):
    """P(kelas 1 | slice) untuk seluruh slice pasien, dgn flip-TTA bila USE_TTA.

    invert_flip=True (stage2 laterality): citra ter-flip → kelas tertukar,
    prob = mean(p_ori, 1 - p_flip) = mean(p[:, 1], p_flip[:, 0]).
    """
    ds = StrokeImageDataset(df, _eval_tf(), cache=False)
    ld = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    p_pos = []
    model.eval()
    for xb, *_ in ld:
        xb = xb.to(device)
        p = torch.softmax(model(xb), dim=1)
        if tta:
            pf = torch.softmax(model(torch.flip(xb, dims=[3])), dim=1)
            p = (p + pf[:, [1, 0]]) / 2 if invert_flip else (p + pf) / 2
        p_pos.extend(p[:, 1].tolist())
    return np.asarray(p_pos)


def aggregate_scores(p, aggregator):
    if aggregator == 'median':
        return float(np.median(p))
    if aggregator == 'max':
        return float(np.max(p)) if len(p) else 0.0
    if aggregator == 'vote':
        return float((p >= 0.5).mean())
    return float(np.mean(p))


def patient_score_for(df, model, aggregator, invert_flip=False):
    """Skor pasien dari agregasi prob slice (pakai SEMUA slice pasien)."""
    return aggregate_scores(slice_pos_prob(df, model, invert_flip), aggregator)


def predict_patient(subject, models, decisions):
    """Pipeline SATU pasien: stage1 → jika LVO lanjut stage2 & 3.

    Keputusan tiap stage memakai AGGREGATOR & THRESHOLD hasil tuning pd VAL (bagian 11).
    """
    res = {'subject': subject}
    d1 = decisions['stage1_lvo']
    df1 = splits[('stage1_lvo', 'test')]
    rows1 = df1[df1['subject'] == subject]
    p1 = patient_score_for(rows1, models['stage1_lvo'], d1['aggregator'])
    res['p_lvo'] = p1
    res['pred_stage1'] = 'LVO' if p1 >= d1['threshold'] else 'Non-LVO'

    if res['pred_stage1'] == 'LVO':
        # ---- Stage 2: Laterality (flip TTA membalik kelas → invert_flip=True) ----
        df2 = splits[('stage2_laterality', 'test')]
        rows2 = df2[df2['subject'] == subject]
        if len(rows2) > 0:
            d2 = decisions['stage2_laterality']
            p2 = patient_score_for(rows2, models['stage2_laterality'], d2['aggregator'], invert_flip=True)
            res['p_right'] = p2
            res['pred_laterality'] = 'Right' if p2 >= d2['threshold'] else 'Left'
        else:
            res['pred_laterality'] = res['p_right'] = None

        # ---- Stage 3: Etiologi ----
        df3 = splits[('stage3_etiology', 'test')]
        rows3 = df3[df3['subject'] == subject]
        if len(rows3) > 0:
            d3 = decisions['stage3_etiology']
            p3 = patient_score_for(rows3, models['stage3_etiology'], d3['aggregator'])
            res['p_cardio'] = p3
            res['pred_etiology'] = 'Cardioembolic' if p3 >= d3['threshold'] else 'Non-Cardioembolic'
        else:
            res['pred_etiology'] = res['p_cardio'] = None
    else:
        res['pred_laterality'] = res['pred_etiology'] = None
        res['p_right'] = res['p_cardio'] = None
    return res

### 13. Evaluasi Pipeline Level Pasien (prediksi-gate vs oracle-gate)

- **prediksi-gate**: pasien yang *diprediksi* LVO oleh stage 1 → dievaluasi di stage 2 & 3
  (skenario nyata saat dipakai).
- **oracle-gate**: pasien yang *benar* LVO (ground truth) → dievaluasi di stage 2 & 3.
  Selisih keduanya menunjukkan dampak salah-klasifikasi stage 1 (kesalahan yang *dapat dicegah*).

In [ ]:
# ===== Evaluasi pipeline end-to-end pada SEMUA pasien test (level pasien) =====
test_subj = splits[('stage1_lvo', 'test')]['subject'].unique()
pipe_rows = [predict_patient(s, models, DECISIONS) for s in test_subj]
pipe = pd.DataFrame(pipe_rows)

# GT per pasien (dari fixed split)
gt1 = splits[('stage1_lvo', 'test')].drop_duplicates('subject').set_index('subject')['label']
gt2 = splits[('stage2_laterality', 'test')].drop_duplicates('subject').set_index('subject')['label']
gt3 = splits[('stage3_etiology', 'test')].drop_duplicates('subject').set_index('subject')['label']
pipe['gt_lvo'] = pipe['subject'].map(gt1)
pipe['gt_lat'] = pipe['subject'].map(gt2)
pipe['gt_etio'] = pipe['subject'].map(gt3)

def fmt_m(m, n):
    return (f'acc={m["accuracy"]:.3f} prec={m["precision"]:.3f} rec={m["recall"]:.3f} '
            f'f1={m["f1"]:.3f} auc={m["auc"]:.3f} (n={n})')

print('===== PIPELINE LEVEL PASIEN — TEST (1x pakai) =====')
# Stage 1: semua pasien
m1 = compute_metrics(pipe['gt_lvo'].tolist(),
                     (pipe['pred_stage1'] == 'LVO').astype(int).tolist(),
                     pipe['p_lvo'].tolist())
print(f'  Stage 1 (prediksi-gate, semua test)   : {fmt_m(m1, len(pipe))}')

def stage_metrics(pred_col, gt_col, p_col, mask):
    sub = pipe[mask].dropna(subset=[pred_col, gt_col])
    if len(sub) == 0:
        return None
    gt = sub[gt_col].astype(int)
    pred = (sub[pred_col].isin(['Right', 'Cardioembolic'])).astype(int)
    return compute_metrics(gt.tolist(), pred.tolist(), sub[p_col].astype(float).tolist()), len(sub)

# Stage 2 & 3 — prediksi-gate (skenario pemakaian nyata)
for stage, (pred_col, gt_col, p_col) in {
        'stage2_laterality': ('pred_laterality', 'gt_lat', 'p_right'),
        'stage3_etiology': ('pred_etiology', 'gt_etio', 'p_cardio')}.items():
    r = stage_metrics(pred_col, gt_col, p_col, pipe['pred_stage1'] == 'LVO')
    print(f'  {stage:22s} (PREDIKsi-gate, diprediksi LVO): '
          + (fmt_m(*r) if r else 'n=0'))

# Stage 2 & 3 — oracle-gate (pasien yang BENAR LVO): dampak salah-klasifikasi stage1 terlihat
print('  --- oracle-gate (pasien yg BENAR LVO) utk komparasi ---')
for stage, (pred_col, gt_col, p_col) in {
        'stage2_laterality': ('pred_laterality', 'gt_lat', 'p_right'),
        'stage3_etiology': ('pred_etiology', 'gt_etio', 'p_cardio')}.items():
    r = stage_metrics(pred_col, gt_col, p_col, pipe['gt_lvo'] == 1)
    print(f'  {stage:22s} (ORACLE-gate, GT LVO): '
          + (fmt_m(*r) if r else 'n=0'))

# Ekspor tabel pasien
pipe.to_csv(os.path.join(OUTPUT_DIR, 'patient_pipeline_predictions.csv'), index=False)
display(pipe.head(10))

### 14. Bootstrap Confidence Interval — Metrik Level Pasien

Test stage 2/3 hanya berisi **~20 pasien** → metrik titik sangat bervariasi. Bootstrap
(resampling per subjek) memberi estimasi interval 95% utk metrik level pasien.

In [ ]:
# ===== Bootstrap 95% CI — metrik level pasien (resampling per SUBJEK) =====
def bootstrap_ci(y_true, p, n_boot=2000, thr=0.5, seed=0):
    """Bootstrap F1 & AUC pd data level pasien, threshold operasi = thr."""
    y_true = np.asarray(y_true); p = np.asarray(p, dtype=float)
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_true))
    f1s, aucs = [], []
    for _ in range(n_boot):
        s = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y_true[s])) < 2:
            continue
        f1s.append(f1_score(y_true[s], (p[s] >= thr).astype(int), zero_division=0))
        aucs.append(roc_auc_score(y_true[s], p[s]))
    return ((np.percentile(f1s, 2.5), np.percentile(f1s, 97.5)),
            (np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)), len(y_true))


print('===== Bootstrap 95% CI — metrik level pasien (TEST, prediksi-gate) =====')
ci_rows = []
for stage, pred_col, gt_col, p_col in [
        ('stage1_lvo', 'pred_stage1', 'gt_lvo', 'p_lvo'),
        ('stage2_laterality', 'pred_laterality', 'gt_lat', 'p_right'),
        ('stage3_etiology', 'pred_etiology', 'gt_etio', 'p_cardio')]:
    if stage == 'stage1_lvo':
        sub = pipe.dropna(subset=['gt_lvo'])
    else:
        sub = pipe[pipe['pred_stage1'] == 'LVO'].dropna(subset=[gt_col, pred_col])
        if len(sub) == 0:
            continue
    thr = DECISIONS[stage]['threshold']
    (flo, fhi), (alo, ahi), n = bootstrap_ci(
        sub[gt_col].astype(int).values, sub[p_col].astype(float).values, thr=thr)
    ci_rows.append((stage, n, round(flo, 3), round(fhi, 3), round(alo, 3), round(ahi, 3)))
    print(f'  {stage:22s} (n={n})  F1 95%CI=[{flo:.3f}, {fhi:.3f}]  AUC 95%CI=[{alo:.3f}, {ahi:.3f}]')
pd.DataFrame(ci_rows, columns=['stage', 'n', 'f1_lo', 'f1_hi', 'auc_lo', 'auc_hi']).to_csv(
    os.path.join(OUTPUT_DIR, 'patient_bootstrap_ci.csv'), index=False)

### 15. Visualisasi Prediksi Pipeline

Contoh prediksi end-to-end per pasien pada test set (hijau = benar, merah = salah).

In [ ]:
def plot_patient_predictions(pipe, n=6):
    sample = pipe.sample(min(n, len(pipe)), random_state=42)
    cols = min(3, len(sample))
    rows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, r) in zip(axes, sample.iterrows()):
        df1 = splits[('stage1_lvo', 'test')]
        rows_p = df1[df1['subject'] == r['subject']]
        mid = rows_p.iloc[len(rows_p) // 2]['image_path']
        img = Image.open(mid).convert('RGB')
        ax.imshow(img); ax.axis('off')

        ok1 = r['gt_lvo'] == int(r['pred_stage1'] == 'LVO')
        t = f"GT : {'LVO' if r['gt_lvo'] == 1 else 'Non-LVO'}\nPRED: {r['pred_stage1']} (p={r['p_lvo']:.2f})"
        color = 'green' if ok1 else 'red'
        if r['pred_laterality'] is not None and pd.notna(r['gt_lat']):
            gt2m = 'Right' if r['gt_lat'] == 1 else 'Left'
            ok2 = r['pred_laterality'] == gt2m
            t += f"\n  Lat: GT={gt2m} PRED={r['pred_laterality']} (p={r['p_right']:.2f})"
            color = 'green' if ok1 and ok2 else 'red'
        if r['pred_etiology'] is not None and pd.notna(r['gt_etio']):
            gt3m = 'Cardioembolic' if r['gt_etio'] == 1 else 'Non-Cardioembolic'
            ok3 = r['pred_etiology'] == gt3m
            t += f"\n  Etio: GT={gt3m} PRED={r['pred_etiology']} (p={r['p_cardio']:.2f})"
            color = 'green' if ok1 and ok2 and ok3 else 'red'
        ax.set_title(t, fontsize=8, color=color)
    for ax in axes[len(sample):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()


plot_patient_predictions(pipe)

### 16. Kurva Training

Kurva loss & F1 validasi per backbone pada tiap stage (seed pertama).

In [ ]:
def plot_training_curves(results):
    for stage in results:
        fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
        for bb in results[stage]:
            hist = results[stage][bb][RUN_SEEDS[0]]['history']
            ax[0].plot(hist['train_loss'], label=bb)
            ax[1].plot(hist['val_pat_f1'], label=bb)
        ax[0].set_title(f'{stage} — train loss'); ax[0].legend(fontsize=8)
        ax[1].set_title(f'{stage} — val F1 level pasien'); ax[1].legend(fontsize=8)
        plt.tight_layout(); plt.show()


plot_training_curves(results)

### 17. [KAGGLE] Arsip & Simpan Weights

Cell ini membuat `ncct_v3_weights.zip` di `/kaggle/working/` lalu instruksi persistensi (Save Version / Commit to Dataset).

In [ ]:
# [KAGGLE] 17. Arsip & Simpan Weights (WAJIB dijalankan terakhir)
# Browser boleh ditutup selama training (background run), TAPI agar weights persisten:
#   1. Jalankan cell ini (membuat ncct_v3_weights.zip di /kaggle/working)
#   2. Klik "Save Version → Save & Run All (Commit)" di Kaggle
#   3. Download zip dari tab Output, atau "Commit to Dataset" untuk dipakai sesi berikutnya
#      (attach dataset tsb → isi LOAD_WEIGHTS_DIR di cell konfigurasi).
kaggle_session_status()
zip_path = zip_kaggle_outputs()
print()
print('NEXT (di Kaggle):')
print('  1. Save Version → Save & Run All (Commit)')
print('  2. Buka tab Output → download ncct_v3_weights.zip, ATAU: Output → ... → Commit to Dataset')
print('  3. Sesi berikutnya: Add Input dataset weights → isi LOAD_WEIGHTS_DIR=/kaggle/input/<slug> → RESUME=True')
print()
print('Ambil output via CLI lokal:  kaggle kernels output <user>/<slug> -p ./ncct_v3_out')
